# Корреляционный анализ и проверка гипотез о связях в данных киберинцидентов

**Рабочая тетрадь студента**  
Продолжительность: 80 минут. Все содержательные выводы формулируйте в контексте работы SOC.

## Правила анализа

1. Сначала строится график, затем выбирается коэффициент или критерий.
2. Корреляция не доказывает причинность.
3. Числовой код категории не превращает номинальный признак в количественный.
4. p-value показывает статистическую значимость, а коэффициент связи — практическую силу эффекта.

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from itertools import combinations

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)

DATA_PATH = 'cyber_correlation_practice.csv'
df = pd.read_csv(DATA_PATH, parse_dates=['timestamp'])

normal = df[df['attack_type'] == 'Normal'].copy()
normal_clean = normal[normal['response_time_ms'] <= 1000].copy()
print('Размер:', df.shape, '| нормальных событий:', len(normal), '| после фильтра:', len(normal_clean))
df.head()

## Вспомогательные функции

In [ ]:
def strength(r):
    a = abs(r)
    return ('очень слабая' if a < .2 else 'слабая' if a < .4 else 'умеренная' if a < .6 else 'сильная' if a < .8 else 'очень сильная')

def correlation_ratio(categories, values):
    data = pd.DataFrame({'cat': categories, 'y': values}).dropna()
    grand = data['y'].mean()
    ss_between = sum(len(g) * (g['y'].mean() - grand) ** 2 for _, g in data.groupby('cat', observed=True))
    ss_total = ((data['y'] - grand) ** 2).sum()
    return np.sqrt(ss_between / ss_total) if ss_total else 0.0

def partial_corr(data, x, y, z):
    r_xy = data[x].corr(data[y])
    r_xz = data[x].corr(data[z])
    r_yz = data[y].corr(data[z])
    return (r_xy-r_xz*r_yz)/np.sqrt((1-r_xz**2)*(1-r_yz**2))

def cramers_v(table):
    chi2 = stats.chi2_contingency(table, correction=False)[0]
    n = table.to_numpy().sum()
    r, k = table.shape
    return np.sqrt(chi2 / (n * min(r-1, k-1)))


## Задание 1. Подготовка данных и выбор методов (2 балла)

1. Выведите размер, типы столбцов, число пропусков и дубликатов.
2. Разделите признаки на количественные, бинарные и многокатегориальные.
3. Для четырёх пар укажите подходящий метод: `requests_per_min–cpu_load_percent`, `attack_type–severity_level`, `incident_confirmed–detection_time_sec`, `attack_type–data_exfil_mb`.
4. Объясните, почему связи между количественными признаками сначала исследуются на нормальном поведении.

In [ ]:
# аудит качества и таблица выбора методов
method_choice = pd.DataFrame({'pair': [], 'method': [], 'reason': []})
method_choice

## Задание 2. Парная корреляция количественных признаков (5 баллов)

На `normal_clean` постройте четыре диаграммы рассеяния: линейная, нелинейная монотонная, U-образная и пара без связи. Для каждой пары вычислите коэффициенты Пирсона, Спирмена и Кендалла. Постройте корреляционную матрицу Пирсона. Затем сравните коэффициенты для `payload_size_bytes–response_time_ms` до и после удаления экстремальных значений. Сформулируйте вывод о форме, направлении, силе связи и чувствительности к выбросам.

In [ ]:
pairs = [('requests_per_min','cpu_load_percent'), ('payload_size_bytes','response_time_ms'),
         ('requests_per_min','network_jitter_ms'), ('source_port','cpu_load_percent')]
# диаграммы рассеяния 2×2

In [ ]:
# таблица Pearson/Spearman/Kendall для pairs и матрица Пирсона
# сравнение полной выборки и normal_clean для payload–response

## Задание 3. Нелинейная, частная и множественная связь (4 балла)

1. Разбейте `requests_per_min` нормальных событий на 10 интервалов и рассчитайте корреляционное отношение η для `network_jitter_ms`. Сопоставьте η с Пирсоном и Спирменом.
2. Рассчитайте парную корреляцию `cpu_load_percent–packet_rate` и частную корреляцию этой пары при фиксированном `requests_per_min`. Объясните роль третьего фактора.
3. Оцените коэффициент множественной корреляции R между `cpu_load_percent` и набором `requests_per_min`, `payload_size_bytes`, используя линейную регрессию и R = √R².

In [ ]:
# интервалы pd.cut и correlation_ratio
# partial_corr
# множественная корреляция через np.linalg.lstsq или sklearn

## Задание 4. Связь категориальных признаков (4 балла)

Постройте таблицу сопряжённости и нормированную столбчатую диаграмму для `attack_type–severity_level`. Проверьте гипотезу независимости критерием χ² Пирсона, проверьте ожидаемые частоты и рассчитайте V Крамера. Разделите статистическую значимость и силу связи в выводе. Дополнительно для таблицы 2×2 `is_after_hours–incident_confirmed` можно вычислить φ и коэффициент Юла Q.

In [ ]:
# pd.crosstab, stacked bar, chi2_contingency и cramers_v
# Дополнительно: φ и Q Юла для таблицы 2×2

## Задание 5. Количественный и категориальный признаки (5 баллов)

1. Сравните `detection_time_sec` для подтверждённых и неподтверждённых инцидентов: boxplot, ECDF, критерий Манна–Уитни и точечно-бисериальная корреляция.
2. Сравните `data_exfil_mb` по типам атак: boxplot в логарифмической шкале, критерий Краскела–Уоллиса, попарный post-hoc Манна–Уитни с поправкой Холма и η².
3. Напишите вывод SOC из 4–6 предложений, не приписывая корреляции причинный смысл.

In [ ]:
# boxplot/ECDF, mannwhitneyu, pointbiserialr
# kruskal, post-hoc с поправкой Холма, eta²
# Напишите итоговый вывод SOC

## Самопроверка

В каждом выводе должны быть названы: пара признаков, направление или характер связи, оценка силы, статистическая значимость, ограничение интерпретации и практический смысл для SOC.